In [ ]:
pip install jupytext 

  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)

   ------- --------------------------------  2/11 [fastjsonschema]
   ---------- -----------------------------  3/11 [attrs]
   ---------- -----------------------------  3/11 [attrs]
   -------------- -------------------------  4/11 [referencing]
   ------------------ ---------------------  5/11 [markdown-it-py]
   ------------------ ---------------------  5/11 [markdown-it-py]
   ------------------ ---------------------  5/11 [markdown-it-py]
   ------------------ ---------------------  5/11 [markdown-it-py]
   ------------------ ---------------------  5/11 [markdown-it-py]
   --------------------- ------------------  6/11 [mdit-py-plugins]
   --------------------- ------------------  6/11 [mdit-py-plugins]
   --------------------- ------------------  6/11 [mdit-py-plugins]
   ------------------------- --------------  7/11 [jsonschema-specifications]
   ----------------------------- ----------  8/11 [jsonschema]
   -------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
! pip install jupytext

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
! pip install jupytext ipykernel

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
! jupytext --set-formats ipynb,py:percent notebook.ipynb
! jupytext --sync notebook.ipynb

'jupytext' is not recognized as an internal or external command,
operable program or batch file.
'jupytext' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
! jupytext --to py:percent notebook.ipynb

'jupytext' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
pip install pyyaml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 1.7 MB/s eta 0:00:01
   ---------------------------- ----------- 1.3/1.8 MB 2.7 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.5 MB/s  0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# Procesamiento de datos de Diputados (2017 y 2025) y match con ideología del Congreso

Este notebook:

1. Importa los dos datasets de resultados electorales de Diputados (2017 y 2025).
2. Para cada uno, se queda solo con: las dos primeras columnas, la última columna (`Cargo`), y las columnas `Nombres`, `Primer apellido`, `Segundo apellido` (que se concatenan en una sola columna `Name`).
3. Agrega una columna `Period` con el valor `period_4` para el dataset de 2017 y `period_6` para el de 2025.
4. Filtra las filas donde `Cargo == "DIPUTADO"`.
5. Colapsa a una fila por persona (por `Name` + `Period`), ya que cada diputado aparece repetido una vez por comuna.
6. Hace el merge con el dataset central `ideologia_congreso_chile_2002_2026_long_format` usando `Period` y `Name` (permitiendo que un mismo Name+Period pueda parear una o más veces).

**Nota:** Ajusta las rutas de los archivos en la celda de configuración si no coinciden con tu estructura de carpetas local.

## 0. Configuración

In [2]:
import pandas as pd
from pathlib import Path

# --- Ajusta esta ruta a la carpeta donde tienes tus archivos ---
DATA_DIR = Path(r"C:\Users\merri\OneDrive - Universidad de Chile\'26\Seminario de Investigacion\Evidence and estimations")

FILE_2013 = DATA_DIR / "2013_11_Diputados_Datos_Eleccion.xlsx"
FILE_2025 = DATA_DIR / "2025_11_Diputados_Datos_Eleccion_v2.xlsx"
FILE_IDEOLOGIA = DATA_DIR / "ideologia_congreso_chile_2002_2026_long_format.csv"

# Nombre de la hoja donde están los resultados por comuna (confirmado en ambos archivos)
SHEET_NAME = "Votación por comuna"

# Los encabezados reales están en la fila 7 del archivo (índice 6, 0-based)
HEADER_ROW = 6

# Columnas de nombre a concatenar, en orden
NAME_COLS = ["Nombres", "Primer apellido", "Segundo apellido"]

# Columna que marca el cargo, y el valor que nos interesa
CARGO_COL = "Cargo"
CARGO_VALUE = "DIPUTADO"

## 1. Función de importación y limpieza

Se define una función reutilizable que:
- Lee el Excel usando la fila de encabezado correcta.
- Se queda solo con las 2 primeras columnas + `Nombres`, `Primer apellido`, `Segundo apellido` + `Cargo`.
- Crea la columna `Period`.
- Filtra por `Cargo == "DIPUTADO"`.
- Concatena los nombres en una columna `Name`.
- Colapsa a una fila por persona (`Name` + `Period`).

In [4]:
def cargar_dataset_diputados(filepath: Path, period_value: str, sheet_name: str = SHEET_NAME) -> pd.DataFrame:
    """
    Importa un dataset de resultados de Diputados y lo deja listo para el merge.

    Pasos:
      1. Lee la hoja indicada, usando la fila 7 (índice 6) como encabezado.
      2. Selecciona: las 2 primeras columnas + Nombres/Primer apellido/Segundo apellido + la última columna (Cargo).
      3. Agrega la columna 'Period' con el valor entregado (period_4 / period_6).
      4. Filtra solo las filas donde Cargo == 'DIPUTADO'.
      5. Concatena Nombres + Primer apellido + Segundo apellido en una columna 'Name'.
      6. Colapsa a una fila por persona (Name + Period), ya que cada diputado
         aparece repetido una vez por cada comuna de su distrito.
    """
    # 1. Leer el archivo completo con los encabezados correctos
    df_raw = pd.read_excel(filepath, sheet_name=sheet_name, header=HEADER_ROW)

    # 2. Identificar las 2 primeras columnas y la última columna del archivo ORIGINAL
    primeras_dos = list(df_raw.columns[:2])
    ultima_col = df_raw.columns[-1]

    columnas_necesarias = primeras_dos + NAME_COLS + [ultima_col]
    faltantes = [c for c in columnas_necesarias if c not in df_raw.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas esperadas en {filepath.name}: {faltantes}")

    df = df_raw[columnas_necesarias].copy()

    # 3. Agregar columna Period
    df["Period"] = period_value

    # 4. Filtrar solo DIPUTADO en la última columna (Cargo)
    df = df[df[ultima_col].astype(str).str.strip().str.upper() == CARGO_VALUE]

    # 5. Concatenar nombres -> columna Name
    df["Name"] = (
        df[NAME_COLS]
        .fillna("")
        .astype(str)
        .apply(lambda row: " ".join(x.strip() for x in row if x.strip() != ""), axis=1)
        .str.strip()
        .str.upper()
    )

    # Reordenar columnas: primeras dos originales, Name, Period, Cargo
    df = df[primeras_dos + ["Name", "Period", ultima_col]]

    # 6. Colapsar a una fila por persona (Name + Period)
    #    Cada diputado aparece repetido una vez por comuna; nos quedamos con un registro único.
    df_unico = df.drop_duplicates(subset=["Name", "Period"]).reset_index(drop=True)

    return df_unico

## 2. Cargar los dos datasets de Diputados

In [5]:
df_2013 = cargar_dataset_diputados(FILE_2013, period_value="period_4")
df_2025 = cargar_dataset_diputados(FILE_2025, period_value="period_6")

print("2013 ->", df_2013.shape)
display(df_2013.head())

print("2025 ->", df_2025.shape)
display(df_2025.head())

2013 -> (120, 5)


,Nro.Región,Región,Name,Period,Cargo
0,1,DE TARAPACA,HUGO GUTIERREZ GALVEZ,period_4,DIPUTADO
1,1,DE TARAPACA,RENZO TRISOTTI MARTINEZ,period_4,DIPUTADO
2,2,DE ANTOFAGASTA,MARCOS ESPINOSA MONARDES,period_4,DIPUTADO
3,2,DE ANTOFAGASTA,FELIPE WARD EDWARDS,period_4,DIPUTADO
4,2,DE ANTOFAGASTA,MARCELA HERNANDO PEREZ,period_4,DIPUTADO


2025 -> (155, 5)


,Nro.Región,Región,Name,Period,Cargo
0,1,DE TARAPACA,CARLOS CARVAJAL GALLARDO,period_6,DIPUTADO
1,1,DE TARAPACA,XIMENA NARANJO PINTO,period_6,DIPUTADO
2,1,DE TARAPACA,ALVARO JOFRE CACERES,period_6,DIPUTADO
3,2,DE ANTOFAGASTA,SEBASTIAN VIDELA CASTILLO,period_6,DIPUTADO
4,2,DE ANTOFAGASTA,JAIME ARAYA GUERRERO,period_6,DIPUTADO


## 3. Unir ambos datasets de Diputados en uno solo

In [4]:
df_diputados = pd.concat([df_2013, df_2025], ignore_index=True)
print("Diputados combinados ->", df_diputados.shape)
display(df_diputados.head())
display(df_diputados["Period"].value_counts())

Diputados combinados -> (275, 5)


,Nro.Región,Región,Name,Period,Cargo
0,1,DE TARAPACA,HUGO GUTIERREZ GALVEZ,period_4,DIPUTADO
1,1,DE TARAPACA,RENZO TRISOTTI MARTINEZ,period_4,DIPUTADO
2,2,DE ANTOFAGASTA,MARCOS ESPINOSA MONARDES,period_4,DIPUTADO
3,2,DE ANTOFAGASTA,FELIPE WARD EDWARDS,period_4,DIPUTADO
4,2,DE ANTOFAGASTA,MARCELA HERNANDO PEREZ,period_4,DIPUTADO


Period
period_6    155
period_4    120
Name: count, dtype: int64

## 4. Cargar el dataset central de ideología del Congreso

In [5]:
df_ideologia = pd.read_csv(FILE_IDEOLOGIA)

print("Ideología congreso ->", df_ideologia.shape)
display(df_ideologia.head())
print(df_ideologia.columns.tolist())

Ideología congreso -> (2574, 18)


,n,Id,Name,Period,dim_IC_low,dim_ideology,dim_IC_high,wnom_ideology,bay_ic_low,bay_ideology,bay_ic_high,quant_parties,party_1,party_2,party_3,party_4,Sex,Birth_date
0,1,74,Jaime Naranjo Ortiz,Periodo_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Masculino,1951-01-12T00:00:00
1,2,74,Jaime Naranjo Ortiz,Periodo_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Masculino,1951-01-12T00:00:00
2,3,74,Jaime Naranjo Ortiz,Periodo_3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Masculino,1951-01-12T00:00:00
3,4,74,Jaime Naranjo Ortiz,Periodo_4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Masculino,1951-01-12T00:00:00
4,5,74,Jaime Naranjo Ortiz,Periodo_5,-0.668336,-0.638761,-0.609186,-0.65922,-0.676355,-0.720717,-0.781002,1.0,PS,NaN,NaN,NaN,Masculino,1951-01-12T00:00:00


['n', 'Id', 'Name', 'Period', 'dim_IC_low', 'dim_ideology', 'dim_IC_high', 'wnom_ideology', 'bay_ic_low', 'bay_ideology', 'bay_ic_high', 'quant_parties', 'party_1', 'party_2', 'party_3', 'party_4', 'Sex', 'Birth_date']


## 5. Match por `Period` y `Name`

Se normaliza el texto de `Name` en ambos datasets (mayúsculas, espacios) antes de aplicar el merge para evitar errores de emparejamiento por formato. El merge es tipo `inner` con `df_ideologia` como base para permitir que un mismo `Name`+`Period` pueda parear una o más veces (por ejemplo si el dataset central tiene múltiples registros por legislatura para la misma persona).

In [ ]:
def normalizar_texto(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        
        .str.upper()
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.replace(r"\s+", " ", regex=True)
    )

df_diputados["Name_norm"] = normalizar_texto(df_diputados["Name"])
df_ideologia["Name_norm"] = normalizar_texto(df_ideologia["Name"])

df_diputados["Period"] = df_diputados["Period"].astype(str).str.strip()
df_ideologia["Period"] = df_ideologia["Period"].astype(str).str.strip()

df_match = df_ideologia.merge(
    df_diputados.drop(columns=["Name"]).rename(columns={"Name_norm": "Name_norm"}),
    on=["Period", "Name_norm"],
    how="left",
    suffixes=("", "_diputados"),
)

In [12]:
# ... existing code ...

def check_names_match_period(file_path):
    with open(file_path, 'r') as file:
        content = file.read()
        # Assuming the names are in a specific format, e.g., "Name: John Doe"
        name_pattern = r"Name: (\w+ \w+)"
        matches = re.findall(name_pattern, content)
        
        if matches:
            for name in matches:
                if not name.endswith('.'):
                    print(f"Name '{name}' does not end with a period.")
                    return False
            print("All names end with a period.")
            return True
        else:
            print("No names found in the file.")
            return False

# Example usage
file_path = 'path/to/your/file.txt'
check_names_match_period(file_path)

# ... existing code ...

FileNotFoundError: [Errno 2] No such file or directory: 'path/to/your/file.txt'

In [7]:
# Count non-NA dim_ideology values by period
display(
    df_match.groupby("Period", dropna=False)["dim_ideology"]
    .count()
    .rename("Non_NA_dim_ideology")
    .reset_index()
)

,Period,Non_NA_dim_ideology
0,Periodo_1,121
1,Periodo_2,123
2,Periodo_3,123
3,Periodo_4,121
4,Periodo_5,163
5,Periodo_6,157


In [14]:
display(df_match.tail(20))

,n,Id,Name,Period,dim_IC_low,dim_ideology,dim_IC_high,wnom_ideology,bay_ic_low,bay_ideology,...,party_1,party_2,party_3,party_4,Sex,Birth_date,Name_norm,Nro.Región,Región,Cargo
2554,2555,1182,Sebastián Videla Castillo,Periodo_5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Masculino,1986-04-29T00:00:00,SEBASTIAN VIDELA CASTILLO,NaN,NaN,NaN
2555,2556,1182,Sebastián Videla Castillo,Periodo_6,-0.879115,-0.907485,-0.935856,-0.561303,-0.547360,-0.523085,...,IND,NaN,NaN,NaN,Masculino,1986-04-29T00:00:00,SEBASTIAN VIDELA CASTILLO,NaN,NaN,NaN
2556,2557,1183,Flor Weisse Novoa,Periodo_1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Femenino,1960-03-27T00:00:00,FLOR WEISSE NOVOA,NaN,NaN,NaN
2557,2558,1183,Flor Weisse Novoa,Periodo_2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Femenino,1960-03-27T00:00:00,FLOR WEISSE NOVOA,NaN,NaN,NaN
2558,2559,1183,Flor Weisse Novoa,Periodo_3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Femenino,1960-03-27T00:00:00,FLOR WEISSE NOVOA,NaN,NaN,NaN
2559,2560,1183,Flor Weisse Novoa,Periodo_4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Femenino,1960-03-27T00:00:00,FLOR WEISSE NOVOA,NaN,NaN,NaN
2560,2561,1183,Flor Weisse Novoa,Periodo_5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Femenino,1960-03-27T00:00:00,FLOR WEISSE NOVOA,NaN,NaN,NaN
2561,2562,1183,Flor Weisse Novoa,Periodo_6,0.843486,0.815116,0.786745,0.621516,0.934305,0.964413,...,UDI,NaN,NaN,NaN,Femenino,1960-03-27T00:00:00,FLOR WEISSE NOVOA,NaN,NaN,NaN
2562,2563,1184,Roberto Celedón Fernández,Periodo_1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Masculino,1947-01-31T00:00:00,ROBERTO CELEDON FERNANDEZ,NaN,NaN,NaN
2563,2564,1184,Roberto Celedón Fernández,Periodo_2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Masculino,1947-01-31T00:00:00,ROBERTO CELEDON FERNANDEZ,NaN,NaN,NaN


## 6. Diagnóstico rápido del match

Revisa cuántos registros del dataset central efectivamente parearon, y cuáles `Name` de diputados no encontraron coincidencia (útil para detectar diferencias de formato en los nombres).

In [1]:

col_en_match = col_diputados_referencia if col_diputados_referencia in df_match.columns else f"{col_diputados_referencia}_diputados"

n_pareados = df_match[col_en_match].notna().sum()
n_total = len(df_match)
print(f"Registros pareados: {n_pareados} / {n_total}")

nombres_diputados_sin_match = set(df_diputados["Name_norm"]) - set(
    df_match.loc[df_match[col_en_match].notna(), "Name_norm"]
)
print(f"Nombres de diputados sin match en ideología: {len(nombres_diputados_sin_match)}")
sorted(nombres_diputados_sin_match)[:20]

NameError: name 'col_diputados_referencia' is not defined

In [8]:
# Step 1: Rename columns
df_diputados.rename(columns={'period_4': 'Periodo_4', 'period_6': 'Periodo_6'}, inplace=True)
df_ideologia.rename(columns={'period_4': 'Periodo_4', 'period_6': 'Periodo_6'}, inplace=True)

# Step 2: Normalize names
def normalize_name(name):
    return name.title()

df_diputados['Name'] = df_diputados['Name'].apply(normalize_name)
df_ideologia['Name'] = df_ideologia['Name'].apply(normalize_name)

# Step 3: Merge DataFrames
merged_df = pd.merge(df_diputados, df_ideologia, on='Period')

# Step 4: Save to CSV (optional)
merged_df.to_csv('merged_diputados_ideologia.csv', index=False)

## 7. Guardar resultado final

In [9]:
OUTPUT_PATH = DATA_DIR / "match_ideologia_diputados.xlsx"
df_match.to_excel(OUTPUT_PATH, index=False)
print(f"Guardado en: {OUTPUT_PATH.resolve()}")

Guardado en: C:\Users\merri\OneDrive - Universidad de Chile\'26\Seminario de Investigacion\Evidence and estimations\match_ideologia_diputados.xlsx


In [10]:
# Check for missing values
print(merged_df.isnull().sum())

# Check for duplicate rows
print(merged_df.duplicated().sum())

Nro.Región       0
Región           0
Name_x           0
Period           0
Cargo            0
Name_norm_x      0
n                0
Id               0
Name_y           0
dim_IC_low       0
dim_ideology     0
dim_IC_high      0
wnom_ideology    0
bay_ic_low       0
bay_ideology     0
bay_ic_high      0
quant_parties    0
party_1          0
party_2          0
party_3          0
party_4          0
Sex              0
Birth_date       0
Name_norm_y      0
dtype: int64
0


In [ ]:
# Check data types
print(merged_df.dtypes)
# Check for consistency in data formats (e.g., dates, numbers)
# Example: Check if dates are in the correct format
merged_df['Period'] = pd.to_datetime(merged_df['Period'], errors='coerce')
print(merged_df['Period'].isnull().sum())

Nro.Región         int64
Región            object
Name_x            object
Period            object
Cargo             object
Name_norm_x       object
n                  int64
Id                 int64
Name_y            object
dim_IC_low       float64
dim_ideology     float64
dim_IC_high      float64
wnom_ideology    float64
bay_ic_low       float64
bay_ideology     float64
bay_ic_high      float64
quant_parties    float64
party_1           object
party_2           object
party_3           object
party_4           object
Sex               object
Birth_date        object
Name_norm_y       object
dtype: object
0


In [10]:
# Check if all relevant data is included
# Example: Check if all periods are present in both DataFrames
periods_in_diputados = df_diputados['Period'].unique()
periods_in_ideologia = df_ideologia['Period'].unique()
print("Periods in df_diputados:", periods_in_diputados)
print("Periods in df_ideologia:", periods_in_ideologia)
print("Common periods:", set(periods_in_diputados) & set(periods_in_ideologia))

Periods in df_diputados: ['period_4' 'period_6']
Periods in df_ideologia: ['Periodo_1' 'Periodo_2' 'Periodo_3' 'Periodo_4' 'Periodo_5' 'Periodo_6']
Common periods: set()


In [13]:
# Example: Check if the names are correctly normalized
print(merged_df['Name'].head())


# Example: Check if the data is accurately represented
# Example: Check if the values in 'Periodo_4' and 'Periodo_6' are consistent
print(merged_df[['Periodo_4', 'Periodo_6']].head())

KeyError: 'Name'